[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch01/NN_DL_ch01_CreditMLP/NN_DL_ch01_CreditMLP.ipynb)

# NN_DL_ch01_CreditMLP

**Name:**  NN_DL_ch01_CreditMLP
**Published in:**  NN and Deep Learning with Business Applications
**Author:**  Daniel Traian Pele, ASE Bucharest / IDA
**Submitted:**  2026-03-24
**Description:**  MLP for credit scoring — training, evaluation, ROC, feature importance
**Keywords:**  neural network, deep learning, credit risk, MLP, classification
**See also:**  NN_DL_ch01_Activations


In [ ]:
# Install dependencies (Colab already has most)
# !pip install torch torchvision matplotlib numpy scikit-learn


In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

# ── Reproducibility ──────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Style constants ──────────────────────────────────────────────────────────
MAIN_BLUE = "#1A3A6E"
IDA_RED = "#CD0000"
FOREST = "#2E7D32"
CRIMSON = "#DC3545"

# ── 1. Synthetic credit-scoring data (1000 samples, 20 features) ────────────
n_samples = 1000
feature_names = [
    "Duration", "CreditAmount", "InstallmentRate", "Age",
    "NumExistingCredits", "NumDependents", "Employment", "Savings",
    "CheckingAccount", "Housing", "Purpose", "OtherDebtors",
    "ResidenceSince", "Property", "OtherInstallments", "ForeignWorker",
    "Telephone", "JobSkill", "MonthlyIncome", "DebtToIncome",
]
n_features = len(feature_names)  # 20

X = np.random.randn(n_samples, n_features).astype(np.float32)
# True logits: weighted combination (some features matter more)
true_weights = np.array(
    [0.8, 1.2, -0.5, -0.7, 0.4, 0.2, -0.9, -1.1, 0.6, 0.3,
     0.5, -0.3, 0.1, -0.4, 0.7, -0.2, 0.15, -0.6, 1.0, -0.8],
    dtype=np.float32,
)
logits = X @ true_weights + 0.3 * np.random.randn(n_samples).astype(np.float32)
y = (logits > 0).astype(np.float32)

print(f"Class balance: {y.mean():.2%} positive (good credit)")

# ── 2. Preprocessing ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# ── 3. Model definition ─────────────────────────────────────────────────────
class CreditMLP(nn.Module):
    """Two-hidden-layer MLP for binary credit scoring: 20 → 64 → 32 → 1."""

    def __init__(self, n_in, hidden1=64, hidden2=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden1),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x)


model = CreditMLP(n_features)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ── 4. Training loop ────────────────────────────────────────────────────────
n_epochs = 150
train_losses, test_losses = [], []

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    logits_train = model(X_train_t)
    loss = criterion(logits_train, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        logits_test = model(X_test_t)
        test_losses.append(criterion(logits_test, y_test_t).item())

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3d}/{n_epochs}  "
              f"train_loss={train_losses[-1]:.4f}  test_loss={test_losses[-1]:.4f}")

# ── 5. Evaluation ────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_test_t)).numpy().flatten()
    preds = (probs >= 0.5).astype(int)

acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Test AUC:      {auc:.4f}")

# ── 6. Figure 1 — Training curves ───────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 4))
fig1.patch.set_alpha(0)
ax1.set_facecolor("none")
ax1.plot(train_losses, color=MAIN_BLUE, linewidth=1.5, label="Train loss")
ax1.plot(test_losses, color=IDA_RED, linewidth=1.5, label="Test loss")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("BCE Loss", fontsize=11)
ax1.set_title("Credit MLP — Training Curves", fontsize=13, fontweight="bold",
              color=MAIN_BLUE)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("nn_ch01_credit_training_curves.pdf", bbox_inches="tight", dpi=300,
            transparent=True)
plt.show()
print("Saved: nn_ch01_credit_training_curves.pdf")

# ── 7. Figure 2 — ROC curve ─────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, probs)

fig2, ax2 = plt.subplots(figsize=(5.5, 5))
fig2.patch.set_alpha(0)
ax2.set_facecolor("none")
ax2.plot(fpr, tpr, color=IDA_RED, linewidth=2, label=f"MLP (AUC = {auc:.3f})")
ax2.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1, label="Random")
ax2.set_xlabel("False Positive Rate", fontsize=11)
ax2.set_ylabel("True Positive Rate", fontsize=11)
ax2.set_title("ROC Curve — Credit Scoring MLP", fontsize=13, fontweight="bold",
              color=MAIN_BLUE)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("nn_ch01_credit_roc.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_credit_roc.pdf")

# ── 8. Figure 3 — Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(y_test, preds)
fig3, ax3 = plt.subplots(figsize=(5, 4.5))
fig3.patch.set_alpha(0)
ax3.set_facecolor("none")
disp = ConfusionMatrixDisplay(cm, display_labels=["Bad", "Good"])
disp.plot(ax=ax3, cmap="Blues", colorbar=False)
ax3.set_title("Confusion Matrix — Credit MLP", fontsize=13, fontweight="bold",
              color=MAIN_BLUE)
plt.tight_layout()
plt.savefig("nn_ch01_credit_confusion.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_credit_confusion.pdf")

# ── 9. Figure 4 — Feature importance (gradient-based) ───────────────────────
model.eval()
X_test_grad = torch.tensor(X_test, dtype=torch.float32, requires_grad=True)
logits_out = model(X_test_grad)
logits_out.sum().backward()
importance = X_test_grad.grad.abs().mean(dim=0).numpy()

sorted_idx = np.argsort(importance)
fig4, ax4 = plt.subplots(figsize=(7, 6))
fig4.patch.set_alpha(0)
ax4.set_facecolor("none")
colors = [MAIN_BLUE if importance[i] >= np.median(importance) else "#999999"
          for i in sorted_idx]
ax4.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx], color=colors,
         edgecolor="white", linewidth=0.5)
ax4.set_xlabel("Mean |Gradient|", fontsize=11)
ax4.set_title("Feature Importance — Credit MLP", fontsize=13, fontweight="bold",
              color=MAIN_BLUE)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("nn_ch01_credit_importance.pdf", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
print("Saved: nn_ch01_credit_importance.pdf")

